In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def add_noises(image, gaussian_var=0.1, s_vs_p=0.5, amount=0.05):
    # Add Gaussian noise
    row, col = image.shape
    mean = 0
    sigma = gaussian_var ** 0.5
    gaussian = np.random.normal(mean, sigma, (row, col))
    noisy = image + gaussian * 255
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)

    # Add salt-and-pepper noise
    num_salt = np.ceil(amount * image.size * s_vs_p)
    num_pepper = np.ceil(amount * image.size * (1 - s_vs_p))

    # Salt
    coords = [np.random.randint(0, i-1, int(num_salt)) for i in image.shape]
    noisy[coords[0], coords[1]] = 255

    # Pepper
    coords = [np.random.randint(0, i-1, int(num_pepper)) for i in image.shape]
    noisy[coords[0], coords[1]] = 0

    return noisy.astype(np.uint8)

In [3]:
def alpha_trimmed_filter(image, d=2, kernel_size=3):
    pad = kernel_size // 2
    padded = cv2.copyMakeBorder(image, pad, pad, pad, pad, cv2.BORDER_REPLICATE)
    filtered = np.zeros_like(image)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            neighborhood = padded[i:i+kernel_size, j:j+kernel_size].flatten()
            neighborhood.sort()
            trimmed = neighborhood[d//2:-d//2] if d != 0 else neighborhood
            filtered[i, j] = np.mean(trimmed)

    return filtered.astype(np.uint8)

In [4]:
def adaptive_local_noise_reduction(image, kernel_size=3, noise_var=100):
    pad = kernel_size // 2
    padded = cv2.copyMakeBorder(image, pad, pad, pad, pad, cv2.BORDER_REPLICATE)
    filtered = np.zeros_like(image, dtype=np.float32)

    global_var = np.var(image)
    eta = 0.25  # Adjustment factor

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            neighborhood = padded[i:i+kernel_size, j:j+kernel_size]
            local_mean = np.mean(neighborhood)
            local_var = np.var(neighborhood)

            if local_var > noise_var:
                filtered[i, j] = image[i, j] - eta * (image[i, j] - local_mean)
            else:
                filtered[i, j] = local_mean

    return np.clip(filtered, 0, 255).astype(np.uint8)

In [5]:
def adaptive_median_filter(image, max_window_size=7):
    filtered = image.copy()
    rows, cols = image.shape

    for i in range(rows):
        for j in range(cols):
            window_size = 3
            done = False

            while window_size <= max_window_size:
                pad = (window_size - 1) // 2
                window = image[max(0, i-pad):min(rows, i+pad+1),
                             max(0, j-pad):min(cols, j+pad+1)]

                z_min = np.min(window)
                z_max = np.max(window)
                z_med = np.median(window)

                A1 = z_med - z_min
                A2 = z_med - z_max

                if A1 > 0 and A2 < 0:
                    B1 = image[i, j] - z_min
                    B2 = image[i, j] - z_max
                    if B1 > 0 and B2 < 0:
                        filtered[i, j] = image[i, j]
                    else:
                        filtered[i, j] = z_med
                    done = True
                    break
                else:
                    window_size += 2

            if not done:
                filtered[i, j] = z_med

    return filtered.astype(np.uint8)

In [ ]:
# Example usage
if __name__ == "__main__":
    # Load image
    image = cv2.imread('/content/HappyFish.jpg', cv2.IMREAD_GRAYSCALE)

    # Add noise
    noisy_image = add_noises(image, gaussian_var=0.05, amount=0.1)

    # Apply filters
    alpha_filtered = alpha_trimmed_filter(noisy_image, d=2, kernel_size=3)
    adaptive_local_filtered = adaptive_local_noise_reduction(noisy_image, kernel_size=3)
    adaptive_median_filtered = adaptive_median_filter(noisy_image, max_window_size=7)

    # Display results
    plt.figure(figsize=(15, 10))

    plt.subplot(221), plt.imshow(noisy_image, cmap='gray')
    plt.title('Noisy Image'), plt.axis('off')

    plt.subplot(222), plt.imshow(alpha_filtered, cmap='gray')
    plt.title('Alpha-Trimmed Filter'), plt.axis('off')

    plt.subplot(223), plt.imshow(adaptive_local_filtered, cmap='gray')
    plt.title('Adaptive Local Filter'), plt.axis('off')

    plt.subplot(224), plt.imshow(adaptive_median_filtered, cmap='gray')
    plt.title('Adaptive Median Filter'), plt.axis('off')

    plt.show()

AttributeError: 'NoneType' object has no attribute 'shape'

In [10]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

def add_noises(image, gaussian_var=0.05, amount=0.1):
    row, col = image.shape
    mean = 0
    sigma = gaussian_var ** 0.5
    gauss = np.random.normal(mean, sigma, (row, col))
    noisy = image + gauss * 255
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    return noisy

def alpha_trimmed_filter(image, d, kernel_size):
    padded_image = np.pad(image, kernel_size // 2, mode='constant', constant_values=0)
    filtered_image = np.zeros_like(image)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            kernel = padded_image[i:i+kernel_size, j:j+kernel_size].flatten()
            kernel.sort()
            trimmed_kernel = kernel[d//2:-d//2]
            filtered_image[i, j] = np.mean(trimmed_kernel)
    return filtered_image

def adaptive_local_noise_reduction(image, kernel_size):
    mean_local = cv2.blur(image, (kernel_size, kernel_size))
    var_local = cv2.blur(image**2, (kernel_size, kernel_size)) - mean_local**2
    var_noise = np.var(image - mean_local)
    filtered_image = image - (var_noise / (var_local + var_noise)) * (image - mean_local)
    return np.clip(filtered_image, 0, 255).astype(np.uint8)

def adaptive_median_filter(image, max_window_size):
    def get_median(img, i, j, window_size):
        half_size = window_size // 2
        window = img[max(i-half_size, 0):min(i+half_size+1, img.shape[0]), max(j-half_size, 0):min(j+half_size+1, img.shape[1])]
        return np.median(window)
    
    filtered_image = np.copy(image)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            window_size = 3
            while window_size <= max_window_size:
                median = get_median(image, i, j, window_size)
                if median != 0 and median != 255:
                    filtered_image[i, j] = median
                    break
                window_size += 2
    return filtered_image

# Example usage
if __name__ == "__main__":
    # Load image
    image = cv2.imread('/content/HappyFish.jpg', cv2.IMREAD_GRAYSCALE)

    # Add noise
    noisy_image = add_noises(image, gaussian_var=0.05, amount=0.1)

    # Apply filters
    alpha_filtered = alpha_trimmed_filter(noisy_image, d=2, kernel_size=3)
    adaptive_local_filtered = adaptive_local_noise_reduction(noisy_image, kernel_size=3)
    adaptive_median_filtered = adaptive_median_filter(noisy_image, max_window_size=7)

    # Display results
    plt.figure(figsize=(15, 10))

    plt.subplot(221), plt.imshow(noisy_image, cmap='gray')
    plt.title('Noisy Image'), plt.axis('off')

    plt.subplot(222), plt.imshow(alpha_filtered, cmap='gray')
    plt.title('Alpha-Trimmed Filter'), plt.axis('off')

    plt.subplot(223), plt.imshow(adaptive_local_filtered, cmap='gray')
    plt.title('Adaptive Local Filter'), plt.axis('off')

    plt.subplot(224), plt.imshow(adaptive_median_filtered, cmap='gray')
    plt.title('Adaptive Median Filter'), plt.axis('off')

    plt.show()


AttributeError: 'NoneType' object has no attribute 'shape'